## 1. Setup & Data Loading {#setup--data-loading}

Import required libraries and load the feature-selected dataset and baseline models from previous steps.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import joblib

# Import machine learning libraries
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.neural_network import MLPClassifier
from scipy.stats import randint, uniform
import matplotlib.pyplot as plt
import seaborn as sns

### Load Processed Data

In [ ]:
# Define data directory
current_user = os.getenv('USER')
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")

# Load data and models
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

# Load 'model_comparison_analysis.pkl'
model_data = joblib.load(PROCESSED_DIR / 'model_comparison_analysis.pkl')

## 2. Model Selection {#model-selection}

Based on the model comparison results, select the top performing models for hyperparameter tuning.

In [ ]:
# Extract models to tune from the loaded data
cv_results = model_data['cv_results']
models_to_tune = model_data['final_models']

print("Models selected for hyperparameter tuning:")
for name in models_to_tune.keys():
    print(f"- {name}")

In [ ]:
# Exclude decision tree due to poor performance during comparison
if 'DecisionTree' in models_to_tune:
    del models_to_tune['DecisionTree']

## 3. Hyperparameter Search Configuration {#hyperparameter-search-configuration}

Define hyperparameter search spaces for each selected model.
We will try two methods: Grid Search and Random Search, so each model two lists are defined (one for each method). 

### 3.1 Random Forest Hyperparameters {#random-forest-hyperparameters}

In [ ]:
# Random Forest hyperparameter grid
rf_param_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'class_weight': ['balanced', None]
}

# Random search distributions for Random Forest
rf_param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': [10, 20, 30, 40, 50, None],
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5),
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'class_weight': ['balanced', None]
}

print("Random Forest hyperparameter search space defined.")

### 3.2 XGBoost Hyperparameters {#xgboost-hyperparameters}

In [ ]:
# XGBoost hyperparameter grid
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 4, 5, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

# Random search distributions for XGBoost
xgb_param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 8),
    'learning_rate': uniform(0.01, 0.3),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'reg_alpha': uniform(0, 2),
    'reg_lambda': uniform(1, 3)
}

print("XGBoost hyperparameter search space defined.")

### 3.3 Logistic Regression Hyperparameters {#logistic-regression-hyperparameters}

In [ ]:
# Logistic Regression hyperparameter grid
lr_param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2', 'elasticnet'],
    'solver': ['liblinear', 'saga'],
    'class_weight': ['balanced', None]
}

# Random search distributions for Logistic Regression
lr_param_dist = {
    'C': uniform(0.001, 100),
    'penalty': ['l1', 'l2', 'elasticnet'],
    'solver': ['liblinear', 'saga'],
    'class_weight': ['balanced', None]
}

print("Logistic Regression hyperparameter search space defined.")

### 3.4 Neural Network Hyperparameters {#neural-network-hyperparameters}

In [ ]:
# Neural Network (MLPClassifier) hyperparameter grid
nn_param_grid = {
    'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 100), (200, 100)],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'lbfgs', 'sgd'],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': [1000, 2000]
}

# Random search distributions for Neural Network
nn_param_dist = {
    'hidden_layer_sizes': [(50,), (100,), (150,), (100, 50), (100, 100), (150, 100), (200, 100), (100, 50, 25)],
    'activation': ['relu', 'tanh', 'logistic'],
    'solver': ['adam', 'lbfgs', 'sgd'],
    'alpha': uniform(0.0001, 0.1),
    'learning_rate': ['constant', 'adaptive'],
    'max_iter': randint(1000, 3001),
    'early_stopping': [True, False],
    'validation_fraction': uniform(0.1, 0.2)
}

print("Neural Network hyperparameter search space defined.")

## 4. Hyperparameter Optimization {#hyperparameter-optimization}

Perform hyperparameter optimization using different search strategies.

In [ ]:
# Set up cross-validation strategy
cv_strategy = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# Scoring metric
scoring = 'roc_auc'

print(f"Cross-validation strategy: {cv_strategy}")
print(f"Scoring metric: {scoring}")

### 4.1 Grid Search {#grid-search}

Perform exhaustive grid search for each model.

In [ ]:
# Store grid search results
grid_search_results = {}

# Grid search for Random Forest
print("Performing Grid Search for Random Forest...")
rf_grid = GridSearchCV(
    estimator=models_to_tune['RandomForest'],
    param_grid=rf_param_grid,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train, y_train)
grid_search_results['RandomForest'] = rf_grid

print(f"Best RF parameters: {rf_grid.best_params_}")
print(f"Best RF score: {rf_grid.best_score_:.4f}")

In [ ]:
# Grid search for XGBoost
print("Performing Grid Search for XGBoost...")
xgb_grid = GridSearchCV(
    estimator=models_to_tune['XGBoost'],
    param_grid=xgb_param_grid,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train, y_train)
grid_search_results['XGBoost'] = xgb_grid

print(f"Best XGB parameters: {xgb_grid.best_params_}")
print(f"Best XGB score: {xgb_grid.best_score_:.4f}")

In [ ]:
# Grid search for Logistic Regression
print("Performing Grid Search for Logistic Regression...")
lr_grid = GridSearchCV(
    estimator=models_to_tune['LogisticRegression'],
    param_grid=lr_param_grid,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1,
    verbose=1
)

lr_grid.fit(X_train, y_train)
grid_search_results['LogisticRegression'] = lr_grid

print(f"Best LR parameters: {lr_grid.best_params_}")
print(f"Best LR score: {lr_grid.best_score_:.4f}")

### 4.2 Random Search {#random-search}

Perform randomized search for comparison and potentially better results.

In [ ]:
# Store random search results
random_search_results = {}
n_iter = 100  # Number of parameter settings sampled

# Random search for Random Forest
print("Performing Random Search for Random Forest...")
rf_random = RandomizedSearchCV(
    estimator=models_to_tune['RandomForest'],
    param_distributions=rf_param_dist,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

rf_random.fit(X_train, y_train)
random_search_results['RandomForest'] = rf_random

print(f"Best RF parameters (Random): {rf_random.best_params_}")
print(f"Best RF score (Random): {rf_random.best_score_:.4f}")

In [ ]:
# Random search for XGBoost
print("Performing Random Search for XGBoost...")
xgb_random = RandomizedSearchCV(
    estimator=models_to_tune['XGBoost'],
    param_distributions=xgb_param_dist,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

xgb_random.fit(X_train, y_train)
random_search_results['XGBoost'] = xgb_random

print(f"Best XGB parameters (Random): {xgb_random.best_params_}")
print(f"Best XGB score (Random): {xgb_random.best_score_:.4f}")

In [ ]:
# Random search for Logistic Regression
print("Performing Random Search for Logistic Regression...")
lr_random = RandomizedSearchCV(
    estimator=models_to_tune['LogisticRegression'],
    param_distributions=lr_param_dist,
    n_iter=n_iter,
    cv=cv_strategy,
    scoring=scoring,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

lr_random.fit(X_train, y_train)
random_search_results['LogisticRegression'] = lr_random

print(f"Best LR parameters (Random): {lr_random.best_params_}")
print(f"Best LR score (Random): {lr_random.best_score_:.4f}")

### 4.3 Bayesian Optimization (Optional) {#bayesian-optimization-optional}

Advanced optimization using Bayesian methods for potentially more efficient search.

In [ ]:
# Bayesian optimization can be implemented here using libraries like:
# - scikit-optimize (skopt)
# - hyperopt
# - optuna

# Example placeholder for Bayesian optimization
print("Bayesian optimization can be implemented here for more advanced tuning.")
print("This would require additional libraries like scikit-optimize, hyperopt, or optuna.")

## 5. Model Evaluation {#model-evaluation}

Evaluate and compare the performance of tuned models.

In [ ]:
# Compare grid search vs random search results
comparison_results = []

for model_name in models_to_tune.keys():
    grid_score = grid_search_results[model_name].best_score_
    random_score = random_search_results[model_name].best_score_
    
    comparison_results.append({
        'Model': model_name,
        'Grid_Search_Score': grid_score,
        'Random_Search_Score': random_score,
        'Best_Method': 'Grid' if grid_score > random_score else 'Random',
        'Score_Difference': abs(grid_score - random_score)
    })

comparison_df = pd.DataFrame(comparison_results)
print("Hyperparameter Tuning Comparison:")
print(comparison_df.round(4))

In [ ]:
# Select best models and evaluate on test set
best_models = {}
test_results = []

for model_name in models_to_tune.keys():
    grid_model = grid_search_results[model_name]
    random_model = random_search_results[model_name]
    
    # Select the better performing search method
    if grid_model.best_score_ > random_model.best_score_:
        best_model = grid_model.best_estimator_
        method = 'Grid Search'
    else:
        best_model = random_model.best_estimator_
        method = 'Random Search'
    
    best_models[model_name] = best_model
    
    # Evaluate on test set
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]
    
    test_results.append({
        'Model': model_name,
        'Method': method,
        'Test_Accuracy': accuracy_score(y_test, y_pred),
        'Test_Precision': precision_score(y_test, y_pred),
        'Test_Recall': recall_score(y_test, y_pred),
        'Test_F1': f1_score(y_test, y_pred),
        'Test_ROC_AUC': roc_auc_score(y_test, y_pred_proba)
    })

test_results_df = pd.DataFrame(test_results)
print("\nTest Set Performance of Tuned Models:")
print(test_results_df.round(4))

In [ ]:
# Visualization of model performance
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# ROC Curves
ax1 = axes[0, 0]
for model_name, model in best_models.items():
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc_score = roc_auc_score(y_test, y_pred_proba)
    ax1.plot(fpr, tpr, label=f'{model_name} (AUC = {auc_score:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', label='Random')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves - Tuned Models')
ax1.legend()
ax1.grid(True)

# Performance metrics comparison
ax2 = axes[0, 1]
metrics = ['Test_Accuracy', 'Test_Precision', 'Test_Recall', 'Test_F1', 'Test_ROC_AUC']
x_pos = np.arange(len(metrics))
width = 0.25

for i, model_name in enumerate(best_models.keys()):
    model_scores = test_results_df[test_results_df['Model'] == model_name][metrics].values[0]
    ax2.bar(x_pos + i*width, model_scores, width, label=model_name)

ax2.set_xlabel('Metrics')
ax2.set_ylabel('Score')
ax2.set_title('Performance Metrics Comparison')
ax2.set_xticks(x_pos + width)
ax2.set_xticklabels([m.replace('Test_', '') for m in metrics], rotation=45)
ax2.legend()
ax2.grid(True)

# Feature importance for the best model
best_model_name = test_results_df.loc[test_results_df['Test_ROC_AUC'].idxmax(), 'Model']
best_model = best_models[best_model_name]

ax3 = axes[1, 0]
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_names = X_train.columns
    indices = np.argsort(importances)[::-1][:20]  # Top 20 features
    
    ax3.bar(range(20), importances[indices])
    ax3.set_xlabel('Features')
    ax3.set_ylabel('Importance')
    ax3.set_title(f'Top 20 Feature Importances - {best_model_name}')
    ax3.set_xticks(range(20))
    ax3.set_xticklabels([feature_names[i] for i in indices], rotation=45, ha='right')
else:
    ax3.text(0.5, 0.5, f'{best_model_name} does not have feature_importances_', 
             ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Feature Importance Not Available')

# Confusion matrix for the best model
ax4 = axes[1, 1]
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4)
ax4.set_xlabel('Predicted')
ax4.set_ylabel('Actual')
ax4.set_title(f'Confusion Matrix - {best_model_name}')

plt.tight_layout()
plt.show()

## 6. Final Model Selection {#final-model-selection}

Select the best performing model based on evaluation metrics.

In [ ]:
# Select final model based on ROC AUC score
best_model_idx = test_results_df['Test_ROC_AUC'].idxmax()
final_model_info = test_results_df.loc[best_model_idx]
final_model_name = final_model_info['Model']
final_model = best_models[final_model_name]

print("\n" + "="*50)
print("FINAL MODEL SELECTION")
print("="*50)
print(f"Selected Model: {final_model_name}")
print(f"Tuning Method: {final_model_info['Method']}")
print(f"Test ROC AUC: {final_model_info['Test_ROC_AUC']:.4f}")
print(f"Test Accuracy: {final_model_info['Test_Accuracy']:.4f}")
print(f"Test Precision: {final_model_info['Test_Precision']:.4f}")
print(f"Test Recall: {final_model_info['Test_Recall']:.4f}")
print(f"Test F1-Score: {final_model_info['Test_F1']:.4f}")

# Get the best hyperparameters
if final_model_info['Method'] == 'Grid Search':
    best_params = grid_search_results[final_model_name].best_params_
else:
    best_params = random_search_results[final_model_name].best_params_

print("\nBest Hyperparameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")

## 7. Model Persistence {#model-persistence}

Save the final optimized model and related artifacts.

In [ ]:
import pickle
import json
from datetime import datetime

# Save the final model
model_filename = f"final_tuned_model_{final_model_name.lower()}.pkl"
model_path = OUT_DIR / model_filename

with open(model_path, 'wb') as f:
    pickle.dump(final_model, f)

print(f"Final model saved to: {model_path}")

# Save model metadata
metadata = {
    'model_name': final_model_name,
    'tuning_method': final_model_info['Method'],
    'best_parameters': best_params,
    'test_performance': {
        'accuracy': float(final_model_info['Test_Accuracy']),
        'precision': float(final_model_info['Test_Precision']),
        'recall': float(final_model_info['Test_Recall']),
        'f1_score': float(final_model_info['Test_F1']),
        'roc_auc': float(final_model_info['Test_ROC_AUC'])
    },
    'training_data_shape': X_train.shape,
    'feature_columns': list(X_train.columns),
    'timestamp': datetime.now().isoformat()
}

metadata_filename = f"model_metadata_{final_model_name.lower()}.json"
metadata_path = OUT_DIR / metadata_filename

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Model metadata saved to: {metadata_path}")

# Save hyperparameter tuning results
results_filename = "hyperparameter_tuning_results.pkl"
results_path = OUT_DIR / results_filename

tuning_results = {
    'grid_search_results': grid_search_results,
    'random_search_results': random_search_results,
    'test_results': test_results_df,
    'comparison_results': comparison_df
}

with open(results_path, 'wb') as f:
    pickle.dump(tuning_results, f)

print(f"Hyperparameter tuning results saved to: {results_path}")

In [ ]:
print("\n" + "="*60)
print("HYPERPARAMETER TUNING COMPLETE")
print("="*60)
print(f"Final model: {final_model_name}")
print(f"Best ROC AUC score: {final_model_info['Test_ROC_AUC']:.4f}")
print(f"Model saved to: {model_path}")
print(f"Ready for production deployment!")
print("="*60)